In [25]:
from filterpy.kalman import KalmanFilter
from project_brain_decoder.train_gru import data_folder
from src.project_brain_decoder.io.nwb_loader import load_nwb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np
import gc

In [2]:
files = list(data_folder.glob("*.nwb"))
train_files = files[:10]
neural_list = []
targets_list = []
X_list = []
y_list = []
neural_scaler = StandardScaler()
targets_scaler = StandardScaler()

In [3]:
for file in train_files:
    loaded_file = load_nwb(file_path=file)
    spiking = loaded_file["neural_spiking_band"]
    threshold = loaded_file["neural_threshold_crossings"]
    neural = np.concatenate([spiking, threshold], axis=1)
    index = loaded_file["target_index_velocity"]
    mrs = loaded_file["target_mrs_velocity"]
    targets = np.column_stack([index, mrs])
    neural_list.append(neural)
    targets_list.append(targets)
neural_all = np.concatenate(neural_list, axis=0)
targets_all = np.concatenate(targets_list, axis=0)
neural_scaler.fit(neural_all)
targets_scaler.fit(targets_all)
del neural_all, targets_all
gc.collect()

4200

In [4]:
for neural, targets in zip(neural_list, targets_list):
    scaled_n = neural_scaler.transform(neural)
    scaled_t = targets_scaler.transform(targets)
    X_list.append(scaled_n)
    y_list.append(scaled_t)
X_train = np.concatenate(X_list)
y_train = np.concatenate(y_list)

In [5]:
# Predicting velocity at t using velocity at t+1 (2x2)
vel_prev = y_train[:-1] # x_{t-1}
vel_curr = y_train[1:] # x_t
F = (vel_curr.T @ vel_prev) @ np.linalg.inv(vel_prev.T @ vel_prev)

In [6]:
# Mapping velocity to neural activity (192x2)
H = (X_train.T @ y_train) @ np.linalg.inv(y_train.T @ y_train)

In [12]:
H.shape

(192, 2)

In [7]:
# Processing noise - residuals of state transition (2x2)
state_res = vel_curr - (F @ vel_prev.T).T
Q = np.cov(state_res.T)

In [8]:
# Observation noise - residuals of observation model (192x192), diagonal
obs_res = X_train - (H @ y_train.T).T
R = np.diag(np.var(obs_res, axis=0))

In [18]:
f = KalmanFilter(dim_x=2, dim_z=192)
f.x = np.array([[0.], [0.]])
f.P = 1000 * np.eye(2)
f.F = F
f.H = H
f.Q = Q
f.R = R

In [19]:
val_file = files[10]
loaded_val = load_nwb(val_file)
spiking_v = loaded_val["neural_spiking_band"]
thresh_v = loaded_val["neural_threshold_crossings"]
neural_val = neural_scaler.transform(np.concatenate([spiking_v, thresh_v], axis=1))

In [23]:
index_vel = loaded_val["target_index_velocity"]
mrs_vel = loaded_val["target_mrs_velocity"]
y_val = np.column_stack([index_vel, mrs_vel])

In [20]:
predictions = []
z = neural_val
T = len(neural_val)

In [21]:
z[t].shape

(192,)

In [22]:
for t in range(T):
    f.predict()
    f.update(z[t].reshape(-1, 1)) # filterpy
    predictions.append(f.x.copy())

In [24]:
preds = np.array(predictions).squeeze() # (T, 2)
preds_unscaled = targets_scaler.inverse_transform(preds)

In [26]:
r2 = r2_score(y_true=y_val, y_pred=preds_unscaled, multioutput="raw_values")
print(f"R² index: {r2[0]:.4f}, R² mrs: {r2[1]:.4f}")

R² index: -0.1567, R² mrs: -1.1185
